# 9C — pharmacology, context, schema-gap, and metadata/features staged batches summary

Focused, reproducible, read-only summary for the N4C replacement lane. This notebook consolidates the accepted staged/non-canonical pharmacology/protein-native tranche, the biological/context/schema-gap tranche, the corrected Sci-Plex candidate/context downgrade, and metadata/features/source-coverage work.

Authoritative coordination inputs are Kanban handoffs/reviews for `t_19516b59` / `t_ee55140a`, `t_de04b319`, `t_63ca49a0` -> `t_587ab15a`, and `t_67465eac` / `t_4de0dfbc`, plus local reports/docs where present. It intentionally replaces stale TODO card `t_2c90f893` without waiting on superseded parent-gated review `t_c91d336a`.

## Safety contract

Default behavior is safe and read-only:

- no downloads;
- no canonical KG writes;
- no GCS writes;
- no heavy Parquet materialization;
- only small local JSON/report/document presence checks are attempted when files already exist.

Optional deep checks are guarded by explicit environment flags:

- `TXGNN_NOTEBOOK_FULL_VALIDATION=1` for heavier local report/parquet audits;
- `TXGNN_NOTEBOOK_ALLOW_GCS_READ=1` for remote `gs://` listing/read checks;
- there is intentionally no flag in this notebook that performs canonical promotion.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists() and (REPO_ROOT.parent / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

FULL_VALIDATION = os.environ.get("TXGNN_NOTEBOOK_FULL_VALIDATION", "0") == "1"
ALLOW_GCS_READ = os.environ.get("TXGNN_NOTEBOOK_ALLOW_GCS_READ", "0") == "1"
ALLOW_CANONICAL_WRITES = False

assert not ALLOW_CANONICAL_WRITES, "This notebook must never enable canonical KG writes."

REPORTS = {
    "opentargets_clinical_drug_evidence": REPO_ROOT / "artifacts" / "reports" / "opentargets_clinical_drug_evidence_report.json",
    "uniprot_disease_protein": REPO_ROOT / "artifacts" / "reports" / "uniprot_disease_associated_protein_validation.json",
    "reactome_pathway_protein": REPO_ROOT / "artifacts" / "reports" / "reactome_pathway_contains_protein_report.json",
    "chembl_molecule_targets_protein": REPO_ROOT / "artifacts" / "reports" / "chembl_molecule_targets_protein_build_summary.json",
    "sciplex_candidate_context": REPO_ROOT / "artifacts/staged/cell_type_molecule_candidate_context_sciplex2_20260622_t_63ca49a0/reports/cell_type_responds_to_molecule_sciplex2_pilot_report.json",
    "paper_dataset_provenance": REPO_ROOT / "artifacts" / "reports" / "paper-dataset-provenance-20260622-t_649cee71" / "reports" / "paper_dataset_provenance_build_report.json",
    "textual_summary_features": REPO_ROOT / "artifacts" / "reports" / "textual-summary-features-20260622-t_3834a45b" / "reports" / "textual_summary_features_summary.json",
    "textual_summary_source_audit": REPO_ROOT / "artifacts" / "reports" / "textual-summary-features-20260622-t_3834a45b" / "reports" / "textual_summary_source_audit.csv",
    "database_gap_matrix": REPO_ROOT / "docs/database_gap_analysis_202606.md",
    "schema_coverage_sweep": REPO_ROOT / "docs/kg_schema_overview.md",
    "source_decision_policy": REPO_ROOT / "docs/source_measure_edge_matrix.md",
    "textual_summary_policy": REPO_ROOT / "docs/textual_summary_features.md",
}
HISTORICAL_LEGACY_REPORT_PATHS = {"note": "Historical provenance only; do not use legacy .omoc reports/staging as current inputs."}


def read_json_if_present(path: Path) -> dict[str, Any] | list[Any] | None:
    if not path.exists():
        return None
    with path.open() as fh:
        return json.load(fh)


report_presence = pd.DataFrame(
    [{"report": name, "path": str(path), "present": path.exists()} for name, path in REPORTS.items()]
)
report_presence

## Consolidated tranche status table

Status vocabulary used here:

- `accepted`: reviewer accepted the staged artifact for the stated scope;
- `staged-only`: useful staged/support artifact, not canonical-ready or not authorized;
- `candidate/context-only`: useful candidate/context table, but no accepted relation edges/evidence;
- `source-gap`: documented absence of a direct source for the requested relation;
- `downstream-deferred`: routed to a later implementation/review lane rather than promoted here;
- `not_promoted`: no canonical KG promotion was performed or authorized.

In [ ]:
tranche_status = pd.DataFrame([
    {
        "tranche": "OpenTargets clinical/safety evidence",
        "relation_or_artifact": "molecule_treats_disease evidence; molecule_contraindicates_disease rejected/not accepted from this source",
        "producer_or_review_cards": "t_ceee5d53; batch t_19516b59; live review t_ee55140a",
        "status": "accepted / staged-only / not_promoted",
        "reviewer_status": "accept positive treatment evidence only; no positive indication -> contraindication misuse",
        "canonical_promotion": "not authorized; not performed",
        "promotion_recommendation": "promote only molecule_treats_disease positive evidence via explicit promotion card; do not promote contraindication evidence from clinical_indication/drug_warning",
    },
    {
        "tranche": "TxGNN/DrugBank drug-combination effect provenance",
        "relation_or_artifact": "molecule_synergizes_molecule evidence backfill for existing edges",
        "producer_or_review_cards": "t_4e12f7c7; batch t_19516b59; live review t_ee55140a",
        "status": "accepted / staged-only / not_promoted",
        "reviewer_status": "accept as conservative provenance for existing TxGNN drug-combination/effect assertions",
        "canonical_promotion": "not authorized; not performed",
        "promotion_recommendation": "do not treat as physical molecular interaction or quantified synergy evidence",
    },
    {
        "tranche": "UniProt source-native disease/protein association",
        "relation_or_artifact": "disease_associated_protein",
        "producer_or_review_cards": "t_7f0cccde; batch t_19516b59; live review t_ee55140a",
        "status": "accepted / staged-only / not_promoted",
        "reviewer_status": "accept UniProtKB DISEASE + humsavar direct protein/isoform endpoints",
        "canonical_promotion": "not authorized; not performed",
        "promotion_recommendation": "preserve UniProt accession/FTId/disease confidence and rejected rows; no gene->protein projection",
    },
    {
        "tranche": "Reactome source-native pathway/protein membership",
        "relation_or_artifact": "pathway_contains_protein",
        "producer_or_review_cards": "t_9d36e82e; batch t_19516b59; live review t_ee55140a",
        "status": "accepted / staged-only / not_promoted",
        "reviewer_status": "accept Reactome UniProt2Reactome_All_Levels direct UniProt endpoints",
        "canonical_promotion": "not authorized; not performed",
        "promotion_recommendation": "explicitly approve all-level pathway semantics before canonical use; no pathway_contains_gene projection",
    },
    {
        "tranche": "ChEMBL source-native molecule/protein targets",
        "relation_or_artifact": "molecule_targets_protein",
        "producer_or_review_cards": "t_84bf3876; batch t_19516b59; live review t_ee55140a",
        "status": "accepted / staged-only / not_promoted",
        "reviewer_status": "accept ChEMBL target_component UniProt rows with unambiguous protein mapping",
        "canonical_promotion": "not authorized; not performed",
        "promotion_recommendation": "keep OpenTargets ENSG MoA out of protein relation; no ambiguous molecule/protein mappings",
    },
    {
        "tranche": "Biological/context/schema-gap batch",
        "relation_or_artifact": "mutation, cell-line assay, disease/tissue, cell-type, Cellosaurus context relations plus documented source gaps",
        "producer_or_review_cards": "t_de04b319 over t_3255672c, t_8ed77c71, t_c2b0803c, t_3ff983e0, t_d468e2dc, t_bb0fb082, t_8e0c701b",
        "status": "accepted staged-only/source-gap mix; one corrected candidate/context-only fix",
        "reviewer_status": "six accepted staged/source-gap producers; Sci-Plex responds_to required downgrade via t_63ca49a0",
        "canonical_promotion": "not authorized; not performed",
        "promotion_recommendation": "respect each source-native and human/product semantics gate before canonical promotion",
    },
    {
        "tranche": "Sci-Plex candidate/context downgrade",
        "relation_or_artifact": "cell_type_responds_to_molecule candidate context table only; 0 accepted edges/evidence",
        "producer_or_review_cards": "t_63ca49a0 -> reviewer t_587ab15a",
        "status": "candidate/context-only / not_promoted",
        "reviewer_status": "accept corrected downgrade; no response/effect metric present in obs metadata",
        "canonical_promotion": "not authorized; not performed",
        "promotion_recommendation": "do not emit cell_type_responds_to_molecule until a real response/effect metric versus control exists",
    },
    {
        "tranche": "Metadata/features/source coverage",
        "relation_or_artifact": "paper/dataset provenance, textual summary feature tables, source decision matrix, schema coverage sweep",
        "producer_or_review_cards": "t_67465eac; live review t_4de0dfbc over t_649cee71, t_3834a45b, t_3d78a14c, t_13b3c3be",
        "status": "accepted / staged-only or review artifact / downstream-deferred / not_promoted",
        "reviewer_status": "accept metadata/features/source-coverage producers as staging/review artifacts",
        "canonical_promotion": "not authorized; not performed",
        "promotion_recommendation": "metadata edges must remain provenance, textual summaries remain feature tables, implementation children stay downstream-deferred",
    },
])
tranche_status

## Pharmacology/protein-native batch accepted via `t_19516b59` / live review `t_ee55140a`

The stale parent-gated card `t_19516b59` was closed as superseded by ungated live review `t_ee55140a`. The live review accepted all five producer outputs as staged/non-canonical artifacts after inspecting producer cards, GCS/local artifacts, source policy docs/code, Parquet samples, endpoint support, and `audit_edge_evidence` where applicable.

In [ ]:
pharmacology_native = pd.DataFrame([
    {
        "tranche": "drug clinical/safety evidence",
        "relation_or_feature_semantics": "OpenTargets clinical_indication supports positive molecule_treats_disease evidence; OpenTargets drug_warning/positive indication are not clean contraindication sources here",
        "source_decision": "accept treatment evidence only; reject contraindication promotion from this batch",
        "script_or_command_paths": "manage_db/stage_opentargets_clinical_drug_evidence.py; tests/test_stage_opentargets_clinical_drug_evidence.py",
        "staged_prefix": "producer-specific GCS staging prefix inspected by t_ee55140a (OpenTargets clinical drug evidence)",
        "counts_and_rejects": "481 treatment evidence rows; 13,654 molecule_treats_disease edges without new evidence; evidence_without_edge=0; missing molecule/disease endpoints=0; drug_warning overlap with canonical contraindication edges=0",
        "validation_review_status": "accepted by t_ee55140a; no canonical writes",
        "canonical_promotion_gate": "promote only positive molecule_treats_disease evidence via explicit card; do not promote molecule_contraindicates_disease evidence from clinical_indication/drug_warning",
    },
    {
        "tranche": "molecule_synergizes_molecule evidence backfill",
        "relation_or_feature_semantics": "TxGNN Dataverse kg.csv drug_drug rows as conservative provenance for existing drug-combination/effect assertions",
        "source_decision": "accept provenance for existing molecule_synergizes_molecule edges; not physical interaction and not quantified synergy",
        "script_or_command_paths": "manage_db/backfill_edge_evidence.py; tests/test_backfill_edge_evidence.py",
        "staged_prefix": "producer-specific GCS staging prefix inspected by t_ee55140a (molecule_synergizes_molecule evidence)",
        "counts_and_rejects": "2,672,628 evidence rows; 2,672,628 supported edge rows; edges_without_evidence=0; evidence_without_edge=0",
        "validation_review_status": "accepted by t_ee55140a; GCS sample copied locally for inspection because DuckDB lacked private GCS credentials",
        "canonical_promotion_gate": "use only as TxGNN/DrugBank drug-combination/effect provenance; do not claim quantified synergy, study, cell-line, dose, effect-size, or p-value context",
    },
    {
        "tranche": "disease_associated_protein source-native pilot",
        "relation_or_feature_semantics": "direct UniProtKB DISEASE comments and UniProt humsavar disease variant rows to protein/isoform endpoints",
        "source_decision": "accept direct UniProt/isoform endpoints; OpenTargets/Reactome/Complex Portal disease fields audited but not materialized",
        "script_or_command_paths": "manage_db/build_uniprot_disease_associated_protein.py; tests/test_build_uniprot_disease_associated_protein.py; manage_db.audit_edge_evidence",
        "staged_prefix": "producer-specific GCS staging prefix inspected by t_ee55140a (disease_associated_protein)",
        "counts_and_rejects": "3,243 edges; 35,839 evidence rows; 26,959 rejected rows; endpoint misses=0; support/orphan evidence=0",
        "validation_review_status": "accepted by t_ee55140a; audit_edge_evidence rerun passed",
        "canonical_promotion_gate": "require no gene->protein projection; preserve UniProt accession/FTId/disease mapping confidence and rejected rows",
    },
    {
        "tranche": "pathway_contains_protein source-native pilot",
        "relation_or_feature_semantics": "Reactome UniProt2Reactome_All_Levels direct UniProt protein membership in pathways",
        "source_decision": "accept direct Reactome UniProt endpoints; no pathway_contains_gene projection",
        "script_or_command_paths": "manage_db/build_reactome_pathway_protein_membership.py; tests/test_build_reactome_pathway_protein_membership.py; docs/reactome_pathway_contains_protein_staged_pilot.md; manage_db.audit_edge_evidence",
        "staged_prefix": "producer-specific GCS staging prefix inspected by t_ee55140a (pathway_contains_protein)",
        "counts_and_rejects": "15,436 edges; 18,068 evidence rows; 141,046 rejected rows; pathway/protein endpoint misses=0; support/orphan evidence=0",
        "validation_review_status": "accepted by t_ee55140a; audit_edge_evidence rerun passed",
        "canonical_promotion_gate": "explicitly approve Reactome all-level pathway semantics before canonical use; do not silently narrow/expand to leaf participants without a new tranche",
    },
    {
        "tranche": "molecule_targets_protein source-native pilot",
        "relation_or_feature_semantics": "ChEMBL target_component UniProt protein target assertions mapped to molecule and protein nodes",
        "source_decision": "accept unambiguous ChEMBL UniProt target_component mappings only; OpenTargets ENSG MoA remains gene-level and excluded",
        "script_or_command_paths": "manage_db/build_chembl_molecule_targets_protein.py; tests/test_build_chembl_molecule_targets_protein.py; docs/source_native_molecule_targets_protein_chembl_pilot.md; manage_db.audit_edge_evidence",
        "staged_prefix": "producer-specific GCS staging prefix inspected by t_ee55140a (molecule_targets_protein)",
        "counts_and_rejects": "2,119 edges; 2,132 evidence rows/source rows; endpoint misses=0; support/orphan evidence=0",
        "validation_review_status": "accepted by t_ee55140a; audit_edge_evidence rerun passed",
        "canonical_promotion_gate": "only ChEMBL target_component UniProt rows with explicit unambiguous protein mapping may enter molecule_targets_protein",
    },
])
pharmacology_native

## Biological/context schema batch via `t_de04b319`

The biological/context live review accepted staged-only and source-gap outputs where source-native semantics were defensible, and created a specific fix for Sci-Plex `cell_type_responds_to_molecule`. This section records relation semantics, counts, rejects, and canonical gates without running builders or writing canonical KG.

In [ ]:
biological_context = pd.DataFrame([
    {
        "producer_card": "t_3255672c",
        "tranche": "mutation genomic direct edges",
        "relation_or_feature_semantics": "mutation_in_gene; mutation_affects_transcript; mutation_overlaps_enhancer from direct genomic/consequence context",
        "source_decision": "accept staged-only",
        "script_or_command_paths": "manage_db/build_staged_mutation_genomic_edges.py; tests/test_build_staged_mutation_genomic_edges.py",
        "staged_prefix": "gs://jouvencekb/kg/v2/staging/source-native-expansion/mutation-genomic-direct-20260622-t_3255672c/",
        "counts_and_rejects": "mutation_in_gene 1,568,719 edges/evidence; mutation_affects_transcript 1,568,719; mutation_overlaps_enhancer 1,664,278; endpoint/evidence support zero-missing",
        "validation_review_status": "accepted staged-only by t_de04b319",
        "canonical_promotion_gate": "needs density policy and review of whether VEP transcriptConsequence targetId fits mutation_in_gene versus consequence target/containment split",
    },
    {
        "producer_card": "t_8ed77c71",
        "tranche": "enhancer_regulates_transcript source audit",
        "relation_or_feature_semantics": "direct enhancer->transcript/ENST/TSS regulation source sought; no gene->all-transcripts expansion allowed",
        "source_decision": "accept as source-gap/no-edge",
        "script_or_command_paths": "GCS source audit JSON; OpenTargets enhancer_to_gene/FANTOM/ENCODE searches",
        "staged_prefix": "source audit artifact only; no edge staging",
        "counts_and_rejects": "direct_source_found=false; 0 staged ENST/TSS edges",
        "validation_review_status": "accepted documented source gap by t_de04b319",
        "canonical_promotion_gate": "do not canonicalize enhancer_regulates_transcript until a direct ENST/TSS endpoint source exists",
    },
    {
        "producer_card": "t_c2b0803c",
        "tranche": "cell-line assays/proteomics",
        "relation_or_feature_semantics": "cell-line essentiality, GDSC molecule response, direct MS/proteomics protein expression",
        "source_decision": "accept staged-only from direct DepMap/GDSC/MS sources; no RNA-to-protein projection",
        "script_or_command_paths": "build_staged_cell_line_assays.py; tests/test_build_staged_cell_line_assays.py",
        "staged_prefix": "producer-specific local/remote validation roots inspected by t_de04b319",
        "counts_and_rejects": "essentiality 1,433,992 edges/evidence; GDSC response 11,040 edges/11,713 evidence; direct MS proteomics 3,083 edges/3,090 evidence; endpoint/support pass",
        "validation_review_status": "accepted staged-only by t_de04b319",
        "canonical_promotion_gate": "needs volume policy for essentiality and broader molecule mapping for GDSC; protein expression only from direct MS/proteomics",
    },
    {
        "producer_card": "t_3ff983e0",
        "tranche": "disease/tissue/phenotype context",
        "relation_or_feature_semantics": "HPA cancer prognostics as disease/tissue context; phenotype_observed_in_tissue and disease_comorbid_disease no-edge/source-gap",
        "source_decision": "accept staged-only with human/product scientific gate",
        "script_or_command_paths": "build_staged_disease_tissue_context.py; tests/test_build_staged_disease_tissue_context.py",
        "staged_prefix": "producer-specific GCS prefix inspected by t_de04b319",
        "counts_and_rejects": "disease_manifests_in_tissue 19 edges/29 evidence; phenotype/comorbidity 0 edges/source-gap; endpoint/evidence support pass",
        "validation_review_status": "accepted staged-only by t_de04b319",
        "canonical_promotion_gate": "human/product confirmation required that TCGA cancer-type columns are direct enough for disease_manifests_in_tissue rather than weaker context relation",
    },
    {
        "producer_card": "t_d468e2dc",
        "tranche": "cell_type hierarchy/tissue/disease context",
        "relation_or_feature_semantics": "CL subtype and tissue context; cell_type_involved_in_disease source-gap",
        "source_decision": "accept staged-only/source-gap mix",
        "script_or_command_paths": "build_cell_type_context_relations.py; tests/test_build_cell_type_context_relations.py",
        "staged_prefix": "producer-specific local/remote report inspected by t_de04b319",
        "counts_and_rejects": "cell_type_subtype_of_cell_type 4,526 edges/evidence; cell_type_found_in_tissue 958 edges/evidence; disease relation source-gap; endpoint/support/duplicates pass",
        "validation_review_status": "accepted staged-only/source-gap by t_de04b319",
        "canonical_promotion_gate": "confirm accepted CL predicates part_of and located_in are semantically suitable for found_in_tissue",
    },
    {
        "producer_card": "t_bb0fb082",
        "tranche": "Cellosaurus cell-line metadata",
        "relation_or_feature_semantics": "cell_line_models_disease and cell_line_derived_from_cell_type metadata/context relations",
        "source_decision": "accept staged-only; direct CL cell-type mappings stronger than disease exact-name mappings",
        "script_or_command_paths": "historical legacy .omoc/scripts/stage_cellosaurus_cell_line_metadata.py (provenance only); current reruns need a reviewed manage_db/artifacts script",
        "staged_prefix": "producer-specific GCS/local validation roots inspected by t_de04b319",
        "counts_and_rejects": "cell_line_models_disease 983 edges/1,218 evidence; cell_line_derived_from_cell_type 65 edges/evidence; endpoint/support pass",
        "validation_review_status": "accepted staged-only by t_de04b319",
        "canonical_promotion_gate": "review exact-name disease mappings and prefer ontology-xref equivalence before canonical promotion",
    },
])
biological_context

## Sci-Plex candidate/context downgrade: `t_63ca49a0` -> `t_587ab15a`

The original Sci-Plex `cell_type_responds_to_molecule` staging was not accepted as a response relation because the source obs metadata lacked a response/effect metric versus control. The fix downgraded the output to candidate/context only: exposure/context rows are useful for downstream inspection, but there are 0 accepted `cell_type_responds_to_molecule` edges and 0 evidence rows.

In [ ]:
sciplex_candidate_context = {
    "producer_fix_card": "t_63ca49a0",
    "review_acceptance_card": "t_587ab15a",
    "relation_or_feature_semantics": "candidate/context exposure rows only; no response/effect assertion",
    "source_decision": "accept corrected downgrade because obs has no response/effect metric versus control; pval_demultiplexing is QC, not response",
    "script_or_command_paths": "artifacts/scripts/stage_cell_type_responds_to_molecule_sciplex2.py; tests/test_stage_cell_type_responds_to_molecule_sciplex2.py",
    "staged_prefix": "gs://jouvencekb/kg/staging/cell_type_molecule_candidate_context_sciplex2_20260622_t_63ca49a0",
    "validation_counts": {
        "candidate_context_rows": 14,
        "rejected_context_rows": 25,
        "edge_rows": 0,
        "evidence_rows": 0,
        "distinct_cell_types": 1,
        "distinct_molecules": 2,
        "edge_without_evidence": 0,
        "evidence_without_edge": 0,
        "endpoint_antijoins": 0,
    },
    "rejection_reason_counts": {"missing_source_chembl_id": 24, "zero_or_missing_dose_value": 4},
    "validation_review_status": "accepted by t_587ab15a after local/GCS readback, source obs audit, pytest, and fresh rematerialization",
    "canonical_promotion_gate": "do not emit cell_type_responds_to_molecule until a real response/effect metric versus control exists",
}

# Include explicit scalar columns for lightweight tests and downstream display.
sciplex_row = {
    **{k: v for k, v in sciplex_candidate_context.items() if k not in {"validation_counts", "rejection_reason_counts"}},
    **sciplex_candidate_context["validation_counts"],
    "rejection_reason_counts": sciplex_candidate_context["rejection_reason_counts"],
}
pd.DataFrame([sciplex_row])

## Metadata/features/source coverage via `t_67465eac` / `t_4de0dfbc`

The metadata/features/source-coverage live review accepted four producers as staging/review artifacts only. Metadata edges must remain provenance/literature/dataset containment records, not biological assertions. Textual summaries are feature tables, not KG causal edges. Source decision matrices and coverage sweeps route downstream work but do not promote canonical KG.

In [ ]:
metadata_features = pd.DataFrame([
    {
        "producer_card": "t_649cee71",
        "tranche": "paper/dataset provenance and citation metadata",
        "relation_or_feature_semantics": "paper_produced_dataset, paper_cites_paper, and dataset containment metadata/provenance relations",
        "source_decision": "accept staged-only metadata/provenance; OpenAlex/EuropePMC/Crossref used; Semantic Scholar audited but not used after 429",
        "script_or_command_paths": "tests/test_build_paper_dataset_provenance.py; local reports paper_dataset_provenance_build_report.json and source_license_access_audit.json",
        "staged_prefix": "gs://jouvencekb/kg/v2/staging/paper-dataset-provenance-20260622-t_649cee71",
        "counts_and_rejects": "paper_produced_dataset=4; paper_cites_paper=16; dataset containment evidence present including empty dataset_contains_disease; dataset_contains_molecule sample=1000",
        "validation_review_status": "accepted by t_4de0dfbc; metadata/literature schemas sampled; no biological assertion conversion found",
        "canonical_promotion_gate": "no canonical promotion without dedicated review; metadata edges must not be treated as biological facts",
    },
    {
        "producer_card": "t_3834a45b",
        "tranche": "textual summary feature tables",
        "relation_or_feature_semantics": "gene/protein/disease/tissue/molecule/pathway textual summary feature Parquets with provenance/license columns",
        "source_decision": "accept OpenTargets/UniProt/GO/UBERON/ChEMBL-style sources with attribution; reject GeneCards scraping; defer DrugBank text and Orphanet; Reactome policy accepted but not staged here",
        "script_or_command_paths": "docs/textual_summary_features.md; tests/test_kg_textual_summary_features.py; tests/test_build_textual_summary_features.py",
        "staged_prefix": "historical legacy .omoc/staging/textual-summary-features-20260622-t_3834a45b/features (provenance only); current reruns should use gs://jouvencekb/kg/staging/textual-summary-features-20260622-t_3834a45b/features or artifacts/staged/...",
        "counts_and_rejects": "six local feature Parquets; gene_textual_summary=212,029; protein_textual_summary pilot=228; feature schema includes feature_key/table/node/source/provenance/license/citation/release",
        "validation_review_status": "accepted by t_4de0dfbc as feature tables, not causal edges",
        "canonical_promotion_gate": "protein coverage is pilot only; no canonical promotion without later review",
    },
    {
        "producer_card": "t_3d78a14c",
        "tranche": "concrete source decision matrix",
        "relation_or_feature_semantics": "routing/spec matrix for external source candidates and follow-up tasks",
        "source_decision": "accept matrix covering CIViC, Monarch, ClinPGx, OMIM, HumanCyc, DGV/SV, BioGRID, IID, InnateDB, IntAct, MINT",
        "script_or_command_paths": "docs/database_gap_analysis_202606.md lines around source decision matrix",
        "staged_prefix": "documentation/review artifact only; no KG staging prefix",
        "counts_and_rejects": "created downstream audit/design cards where value/schema/access were clear: BioGRID t_9c1aadd3, IntAct t_b18bfacb, CIViC t_aeb0c40f, Monarch t_90a2d388, SV design t_97c6a56d; deferred/rejected ClinPGx direct ingestion, OMIM, HumanCyc, IID, InnateDB, standalone MINT",
        "validation_review_status": "accepted by t_4de0dfbc",
        "canonical_promotion_gate": "children remain audit/design/staged-only until each returns review-required evidence; downstream-deferred",
    },
    {
        "producer_card": "t_13b3c3be",
        "tranche": "schema coverage sweep",
        "relation_or_feature_semantics": "coverage map for every docs/kg_schema_overview.md `GCS? no` relation plus duplicate/stale risk flags",
        "source_decision": "accept sweep as coverage/review artifact",
        "script_or_command_paths": "docs/kg_schema_overview.md parse; reviewer independent parse of GCS? no rows",
        "staged_prefix": "documentation/review artifact only; no KG staging prefix",
        "counts_and_rejects": "32 relations with GCS? no; coverage matrix maps all 32 to cards or deferral/policy decisions; cell_type_expresses_protein routed via t_bbc6fe1c/t_5a77f09c",
        "validation_review_status": "accepted by t_4de0dfbc",
        "canonical_promotion_gate": "do not mark overall schema coverage final until batch gates and new cell_type_expresses_protein path complete",
    },
])
metadata_features

## Commands, report paths, and validation evidence

Commands below are recorded for reproducibility. They are not run by default in this notebook. Builder commands may stage artifacts, so use them only from their producer/review cards or a new authorized card. The notebook's own validation remains metadata-only/read-only.

In [ ]:
commands = pd.DataFrame([
    {
        "tranche": "pharmacology/protein-native batch",
        "builder_or_validation_command": "uv run python -m py_compile manage_db/stage_opentargets_clinical_drug_evidence.py manage_db/build_uniprot_disease_associated_protein.py manage_db/build_reactome_pathway_protein_membership.py manage_db/build_chembl_molecule_targets_protein.py manage_db/backfill_edge_evidence.py; uv run --group dev pytest tests/test_stage_opentargets_clinical_drug_evidence.py tests/test_build_uniprot_disease_associated_protein.py tests/test_build_reactome_pathway_protein_membership.py tests/test_build_chembl_molecule_targets_protein.py tests/test_backfill_edge_evidence.py -q",
        "default_in_this_notebook": "not run",
        "evidence": "t_ee55140a reran compile/tests and audit_edge_evidence where applicable: 19 targeted tests passed",
    },
    {
        "tranche": "biological/context schema batch",
        "builder_or_validation_command": "uv run --group dev pytest tests/test_build_staged_mutation_genomic_edges.py tests/test_build_staged_cell_line_assays.py tests/test_build_staged_disease_tissue_context.py tests/test_build_cell_type_context_relations.py tests/test_stage_cell_type_responds_to_molecule_sciplex2.py tests/test_kg_evidence.py tests/test_kg_schema_cleanup.py -q",
        "default_in_this_notebook": "not run",
        "evidence": "t_de04b319 reviewer ran targeted suite: 22 passed; independent Parquet endpoint/evidence-support audit clean for reviewed staged relations",
    },
    {
        "tranche": "Sci-Plex candidate/context downgrade",
        "builder_or_validation_command": "uv run python -m py_compile artifacts/scripts/stage_cell_type_responds_to_molecule_sciplex2.py; uv run --group dev pytest tests/test_stage_cell_type_responds_to_molecule_sciplex2.py -q",
        "default_in_this_notebook": "not run",
        "evidence": "t_587ab15a accepted local/GCS artifacts and fresh rematerialization; 1 passed; 0 edges/evidence",
    },
    {
        "tranche": "metadata/features/source coverage",
        "builder_or_validation_command": "uv run --group dev pytest tests/test_build_paper_dataset_provenance.py tests/test_kg_textual_summary_features.py tests/test_build_textual_summary_features.py -q; independent parse of docs/kg_schema_overview.md for GCS? no rows",
        "default_in_this_notebook": "not run",
        "evidence": "t_4de0dfbc accepted four producers; 10 passed; parse found 32 GCS? no relations; sampled feature/provenance Parquets",
    },
    {
        "tranche": "Notebook structure",
        "builder_or_validation_command": "uv run python -c 'import nbformat; nbformat.validate(nbformat.read(\"reproduce/12_pharmacology_context_metadata_summary.ipynb\", as_version=4))'; uv run --group dev pytest tests/test_notebook_structure.py -q",
        "default_in_this_notebook": "safe metadata validation only",
        "evidence": "run after edits for this card",
    },
])
commands

## Canonical-promotion gates and downstream-deferred items

1. No covered tranche is canonically promoted by this notebook or by the live reviews summarized here.
2. Pharmacology/protein-native artifacts require explicit promotion/apply cards, live canonical endpoint anti-joins, fresh `audit_edge_evidence`, source release/license verification, and relation-specific gates (`molecule_contraindicates_disease` excluded from OpenTargets clinical/drug_warning; synergy is not physical/quantified; no gene->protein projection; Reactome all-level semantics explicit; ChEMBL protein targets only).
3. Biological/context staged artifacts require relation-specific human/product gates for density, VEP semantics, TCGA disease/tissue context, CL tissue predicates, Cellosaurus disease mapping confidence, and direct-source requirements for source-gap/no-edge relations.
4. Sci-Plex is candidate/context-only with 0 accepted response edges/evidence until a real response/effect metric versus control exists.
5. Metadata edges remain provenance/literature/dataset metadata, textual summaries remain feature tables, and source decision/coverage matrix items are downstream-deferred until their child cards complete review.
6. Cross-cutting repo/process gate remains: accepted code/docs must be repaired/applied in a proper txgnn repo branch/PR or equivalent reviewable patch before source-control merge.

In [ ]:
assert set(["accepted / staged-only / not_promoted", "candidate/context-only / not_promoted"]).issubset(set(tranche_status["status"])), "status vocabulary missing expected entries"
assert not ALLOW_CANONICAL_WRITES
assert "0 accepted edges/evidence" in tranche_status.loc[tranche_status["tranche"].str.contains("Sci-Plex"), "relation_or_artifact"].iloc[0]
assert sciplex_candidate_context["validation_counts"]["edge_rows"] == 0
assert sciplex_candidate_context["validation_counts"]["evidence_rows"] == 0
assert "downstream-deferred" in tranche_status.loc[tranche_status["tranche"].str.contains("Metadata"), "status"].iloc[0]
assert "no gene->protein projection" in pharmacology_native.to_string(index=False)

read_only_validation = pd.DataFrame([
    {"check": "canonical writes disabled", "passed": not ALLOW_CANONICAL_WRITES},
    {"check": "no GCS reads unless TXGNN_NOTEBOOK_ALLOW_GCS_READ=1", "passed": not ALLOW_GCS_READ or ALLOW_GCS_READ},
    {"check": "Sci-Plex remains candidate/context-only", "passed": sciplex_candidate_context["validation_counts"]["edge_rows"] == 0 and sciplex_candidate_context["validation_counts"]["evidence_rows"] == 0},
    {"check": "metadata/features include downstream-deferred status", "passed": "downstream-deferred" in tranche_status.loc[tranche_status["tranche"].str.contains("Metadata"), "status"].iloc[0]},
    {"check": "all consolidated rows are not promoted", "passed": tranche_status["status"].str.contains("not_promoted").all()},
])
read_only_validation